# Lumis 13: Calibrate model v2 on phone photos

Rules fixed before training (report notes, 2026-11-11):
1. **Acceptance:** v2 replaces v1 only if, on SCIN validation, **accuracy and macro F1 both improve**, and DermNet
   validation accuracy drops by **no more than 3 points**. Otherwise this notebook stops and v1 stays.
2. **Prior adjustment:** the same as v1 (DermNet training-mix prior, tau 0.75), not re-tuned.
3. **Temperature:** fitted on SCIN validation close-ups (the photo style Lumis will get); out-of-fold for reporting.
4. **Bands:** same rule as v1 (lowest threshold with at least 20 photos reaching 85% for high, 75% for everything
   answered), on SCIN validation. If 85% cannot be reached, there is no high band. If 75% cannot be reached
   (decided by Zan 2026-11-11, before any v2 result): Lumis still shows the most likely condition with a doctor
   recommendation, using the first reachable target in 70%, 65%, 60%, 55%, 50% ("more often right than wrong" is the
   floor). Only if not even 50% is reachable is everything referred.
5. Scabies cap stays.

**Rule 1, added 2026-11-11 after the first v2 run's validation result and before v2b was trained:** v2 must also not
lower SCIN validation **macro sensitivity** or **tinea sensitivity** below v1. Reason: the first v2 run met the
original rule (accuracy 37% -> 64%) only by calling most phone photos eczema; tinea sensitivity fell from 6/19 to
3/19, and tinea treated as eczema gets worse. Recorded as a change made after seeing a validation result; the test
split was still unused.

CPU runtime is fine. Takes about a minute.

In [ ]:
RUN = None             # None = the latest v2 run (split 'v2+scin')
TARGET_HIGH = 0.85
TARGET_ANSWERED = 0.75
FALLBACK_TARGETS = [0.70, 0.65, 0.60, 0.55, 0.50]
MAX_DERMNET_DROP = 0.03
SEED = 42

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/Lumis'

In [ ]:
import os, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score

runs = pd.read_csv(f'{BASE}/exports/runs_log.csv')
if RUN is None:
    RUN = runs[runs.split == 'v2+scin'].iloc[-1]['run']
info = runs[runs.run == RUN].iloc[-1]
RUN_DIR = f'{BASE}/exports/runs/{RUN}'
V1_RUN = info.base_run
cal1 = json.load(open(f'{BASE}/exports/runs/{V1_RUN}/calibration.json'))
CLASSES = cal1['classes']
ADJ = np.array(cal1['logit_adjust'])          # rule 2: same prior adjustment as v1
T1 = cal1['temperature']

s_v2 = np.load(f'{RUN_DIR}/scin_val_logits.npy')
s_v1 = np.load(f'{RUN_DIR}/scin_val_logits_v1.npy')
s_y = np.load(f'{RUN_DIR}/scin_val_labels.npy')
d_v2 = np.load(f'{RUN_DIR}/val_logits.npy')
d_v1 = np.load(f'{BASE}/exports/runs/{V1_RUN}/val_logits.npy')
d_y = np.load(f'{RUN_DIR}/val_labels.npy')
print(f'v2 run {RUN} (from {V1_RUN}) | SCIN validation {len(s_y)} cases | DermNet validation {len(d_y)} photos')
print('Prior adjustment (same as v1):', dict(zip(CLASSES, np.round(ADJ, 3))))

## 1. Acceptance check (rule 1)

In [ ]:
def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

TINEA = CLASSES.index('tinea')

def quality(logits, y):
    pred = (logits + ADJ).argmax(1)
    sens = [float((pred[y == k] == k).mean()) for k in range(len(CLASSES)) if (y == k).any()]
    return {'accuracy': float((pred == y).mean()),
            'macro_f1': float(f1_score(y, pred, average='macro', labels=range(len(CLASSES)), zero_division=0)),
            'macro_sensitivity': float(np.mean(sens)),
            'tinea_sensitivity': float((pred[y == TINEA] == TINEA).mean())}

acc = pd.DataFrame({'v1': {**{f'SCIN {k}': v for k, v in quality(s_v1, s_y).items()},
                           **{f'DermNet {k}': v for k, v in quality(d_v1, d_y).items()}},
                    'v2': {**{f'SCIN {k}': v for k, v in quality(s_v2, s_y).items()},
                           **{f'DermNet {k}': v for k, v in quality(d_v2, d_y).items()}}})
acc['change'] = acc.v2 - acc.v1
display(acc.round(4))
checks = {'SCIN accuracy improves': acc.loc['SCIN accuracy', 'change'] > 0,
          'SCIN macro F1 improves': acc.loc['SCIN macro_f1', 'change'] > 0,
          'SCIN macro sensitivity not lower': acc.loc['SCIN macro_sensitivity', 'change'] >= 0,
          'SCIN tinea sensitivity not lower': acc.loc['SCIN tinea_sensitivity', 'change'] >= 0,
          f'DermNet accuracy drop at most {MAX_DERMNET_DROP:.0%}': acc.loc['DermNet accuracy', 'change'] >= -MAX_DERMNET_DROP}
for k, ok in checks.items():
    print(f'{"ok  " if ok else "FAIL"}  {k}')
accepted = all(checks.values())
print('ACCEPTED: v2 replaces v1' if accepted else 'NOT ACCEPTED: v1 stays (rule 1). Report this result as it is.')
if not accepted:
    json.dump({'run': RUN, 'accepted': False, 'acceptance': acc.round(4).to_dict()},
              open(f'{RUN_DIR}/acceptance.json', 'w'), indent=1)
    raise RuntimeError('Stopped: v2 did not meet the acceptance rule. Do not run the cells below.')

## 2. Temperature (rule 3)

In [ ]:
def ece(probs, y, bins=15):
    conf, correct = probs.max(1), probs.argmax(1) == y
    edges, total = np.linspace(0, 1, bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(total)

def nll(probs, y):
    return float(-np.mean(np.log(probs[np.arange(len(y)), y] + 1e-12)))

def fit_temperature(z, y):
    res = minimize_scalar(lambda lt: nll(softmax(z / np.exp(lt)), y), bounds=(-3, 3), method='bounded')
    return float(np.exp(res.x))

z = s_v2 + ADJ
raw = softmax(z)
oof, fold_T = np.zeros_like(raw), []
for tr, te in KFold(5, shuffle=True, random_state=SEED).split(z):
    T = fit_temperature(z[tr], s_y[tr])
    fold_T.append(T)
    oof[te] = softmax(z[te] / T)
T_FINAL = fit_temperature(z, s_y)
cal_stats = {'temperature': round(T_FINAL, 4), 'temperature_per_fold': [round(t, 3) for t in fold_T],
             'ece_raw': round(ece(raw, s_y), 4), 'ece_calibrated_oof': round(ece(oof, s_y), 4),
             'nll_raw': round(nll(raw, s_y), 4), 'nll_calibrated_oof': round(nll(oof, s_y), 4),
             'dermnet_val_ece_with_this_T': round(ece(softmax((d_v2 + ADJ) / T_FINAL), d_y), 4)}
print(json.dumps(cal_stats, indent=1))

def reliability(ax, probs, y, title, bins=10):
    conf, correct = probs.max(1), probs.argmax(1) == y
    edges = np.linspace(0, 1, bins + 1)
    mids, accs = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            mids.append((lo + hi) / 2); accs.append(correct[m].mean())
    ax.bar(mids, accs, width=1 / bins, edgecolor='black', alpha=0.7)
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_xlabel('model confidence'); ax.set_ylabel('actual accuracy')
    ax.set_title(f'{title}\nECE = {ece(probs, y):.3f}')

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
reliability(ax[0], raw, s_y, 'SCIN validation: raw')
reliability(ax[1], oof, s_y, f'SCIN validation: T = {T_FINAL:.2f} (out-of-fold)')
plt.tight_layout(); plt.savefig(f'{RUN_DIR}/reliability_scin_val.png', dpi=100); plt.show()

## 3. Bands (rule 4)

In [ ]:
conf, correct = oof.max(1), oof.argmax(1) == s_y

rows = []
for t in np.round(np.arange(0.3, 0.95, 0.05), 2):
    m = conf >= t
    rows.append({'threshold': t, 'answered': round(m.mean(), 3),
                 'accuracy_when_answering': round(correct[m].mean(), 3) if m.any() else None, 'n': int(m.sum())})
sel = pd.DataFrame(rows)
display(sel)

def lowest_threshold(target, min_cases=20):
    for t in np.round(np.arange(0.2, 0.99, 0.01), 2):
        m = conf >= t
        if m.sum() >= min_cases and correct[m].mean() >= target:
            return float(t)
    return None

t_high, t_low = lowest_threshold(TARGET_HIGH), lowest_threshold(TARGET_ANSWERED)
answered_target = TARGET_ANSWERED
notes = []
if t_high is None:
    notes.append(f'No threshold reaches {TARGET_HIGH:.0%} with 20+ photos: the high band is empty (never shown).')
if t_low is None:
    for target in FALLBACK_TARGETS:
        t_low = lowest_threshold(target)
        if t_low is not None:
            answered_target = target
            notes.append(f'{TARGET_ANSWERED:.0%} not reachable; fallback target {target:.0%} used (rule 4).')
            break
if t_low is None:
    answered_target = None
    notes.append('Not even 50% reachable with 20+ photos: every result is referred.')
high_from = t_high if t_high is not None else 1.01
refer_below = t_low if t_low is not None else 1.01
high_from = max(high_from, refer_below)
hi, lo = conf >= high_from, conf < refer_below
mid = ~hi & ~lo
bands = {'high_from': high_from, 'moderate_from': refer_below, 'refer_below': refer_below,
         'answered_target_used': answered_target,
         'share': {'high': round(hi.mean(), 3), 'moderate': round(mid.mean(), 3), 'refer': round(lo.mean(), 3)},
         'accuracy': {k: (round(correct[m].mean(), 3) if m.any() else None) for k, m in [('high', hi), ('moderate', mid), ('refer', lo)]},
         'notes': notes}
print(json.dumps(bands, indent=1))
for n in notes:
    print('NOTE:', n)

## 4. Save

In [ ]:
assert accepted, 'v2 was not accepted: nothing to save.'
json.dump({'run': RUN, 'accepted': True, 'acceptance': acc.round(4).to_dict()}, open(f'{RUN_DIR}/acceptance.json', 'w'), indent=1)
out = {'run': RUN, 'backbone': 'efficientnetb0', 'model': 'v2', 'base_run': V1_RUN, 'classes': CLASSES,
       'calibrated_on': 'SCIN validation close-ups', 'targets': {'high': TARGET_HIGH, 'answered': TARGET_ANSWERED},
       'prior_tau': cal1['prior_tau'], 'class_prior': cal1['class_prior'], 'logit_adjust': cal1['logit_adjust'],
       **cal_stats, 'bands': bands, 'accepted': True, 'acceptance': acc.round(4).to_dict()}
json.dump(out, open(f'{RUN_DIR}/calibration.json', 'w'), indent=1)
sel.to_csv(f'{RUN_DIR}/selective_accuracy_scin_val.csv', index=False)
print('Saved', f'{RUN_DIR}/calibration.json')